# 绘制XGBoost-UNet超参数的重要性
- 依赖于study.pkl文件加载的优化对象
- 文件保存路径为'../datas/study.pkl'
- 创建文件为'main_FieldForecast_Unet_optim_xgboost_v6.3.ipynb'

## 加载study文件

In [1]:
import joblib

study = joblib.load("../datas/study.pkl")

d:\Softwares\Anaconda3\envs\cnn-model\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import optuna
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from optuna.importance import FanovaImportanceEvaluator

# 在代码开始处定义
stable_evaluator = FanovaImportanceEvaluator(seed=2222)

# 1. 配置
output_dir = '../figs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

label_map = {
    "lr": "Learning rate",
    "num_models": "Number of models",
    "shrinkage": "Shrinkage"
}

# 2. 预先生成对象
fig_imp_raw = optuna.visualization.plot_param_importances(study, evaluator=stable_evaluator)
fig_c1 = optuna.visualization.plot_contour(study, params=['lr', 'num_models'])
fig_c2 = optuna.visualization.plot_contour(study, params=['num_models', 'shrinkage'])
fig_c3 = optuna.visualization.plot_contour(study, params=['lr', 'shrinkage'])

# 3. 创建布局 (增加 horizontal_spacing 避免大字号重叠)
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "(a) Parameter Importance", 
        "(b) LR vs Num Models",
        "(c) Num Models vs Shrinkage", 
        "(d) LR vs Shrinkage"
    ),
    horizontal_spacing=0.2, 
    vertical_spacing=0.2
)

# --- 子图 (1,1): Importance ---
imp_data = fig_imp_raw.data[0]
new_y = [label_map.get(label, label) for label in imp_data.y]
fig.add_trace(
    go.Bar(x=imp_data.x, y=new_y, orientation='h', 
           marker_color='#4E79A7', marker_line_color='black', marker_line_width=1.5,
           texttemplate='%{x:.2f}', 
           textposition='inside',
           textfont=dict(size=24, color='white')), # 重要性数值
    row=1, col=1
)

# --- 子图迁移辅助函数 ---
def add_contour_to_subplot(source_fig, row, col):
    for trace in source_fig.data:
        if isinstance(trace, go.Contour):
            trace.showscale = (row == 2 and col == 2)
            if trace.showscale:
                trace.colorbar = dict(
                    title="Objective", 
                    thickness=20, 
                    x=1.05,
                    tickfont=dict(size=20)
                )
            fig.add_trace(trace, row=row, col=col)
        elif isinstance(trace, go.Scatter):
            trace.marker.size = 6
            trace.marker.opacity = 0.4
            fig.add_trace(trace, row=row, col=col)

add_contour_to_subplot(fig_c1, 1, 2)
add_contour_to_subplot(fig_c2, 2, 1)
add_contour_to_subplot(fig_c3, 2, 2)

# 4. 统一美化与对数坐标设置
fig.update_layout(
    template='plotly_white',
    width=1400,   # 略微增加宽度容纳大字号标签
    height=1000,
    font=dict(family="Arial", color='black'),
    showlegend=False,
    margin=dict(l=10, r=10, t=80, b=20)
)

# 坐标轴配置字典
# (行, 列, X轴标题, Y轴标题, X轴是否对数, Y轴是否对数)
axis_configs = [
    (1, 1, "Importance Score", "", False, False),
    (1, 2, label_map["lr"], label_map["num_models"], True, False),  # LR在X轴，设为对数
    (2, 1, label_map["num_models"], label_map["shrinkage"], False, False),
    (2, 2, label_map["lr"], label_map["shrinkage"], True, False)    # LR在X轴，设为对数
]

# 修改循环部分，确保第一张图的 Y 轴是类别型 (category)
for r, c, xtitle, ytitle, x_log, y_log in axis_configs:
    # X 轴美化
    fig.update_xaxes(
        title_text=xtitle, 
        row=r, col=c, 
        title_font=dict(size=26),
        tickfont=dict(size=24),
        # 强制确保第一张图 X 轴是线性，后面涉及到 LR 的才是对数
        type='log' if (x_log and r > 0) else 'linear', 
        showline=True, linewidth=2, linecolor='black', mirror=True, ticks='outside'
    )
    
    # Y 轴美化
    if r == 1 and c == 1:
        # 第一张图是条形图，Y 轴必须是 category 类型
        fig.update_yaxes(
            title_text="", 
            row=r, col=c, 
            tickfont=dict(size=24),
            type='category', 
            showline=True, linewidth=2, linecolor='black', mirror=True
        )
    else:
        fig.update_yaxes(
            title_text=ytitle, 
            row=r, col=c, 
            title_font=dict(size=26), 
            tickfont=dict(size=24),
            type='log' if y_log else 'linear',
            showline=True, linewidth=2, linecolor='black', mirror=True, ticks='outside'
        )

# 修正子图标题 (a,b,c,d) 的字号
for i in fig['layout']['annotations']:
    i['font'] = dict(size=28, color='black', family="Arial Black")

# 5. 保存高清图
save_path = os.path.join(output_dir, "combined_analysis_academic.png")
# 使用 scale=3 导出，最终像素约为 3600x3000，完美适配高质量打印
fig.write_image(save_path, scale=3, engine="kaleido")

print(f"论文级图表已保存至: {save_path}")
fig.show()

C:\Users\zhihe\AppData\Local\Temp\ipykernel_78020\3769503337.py:133: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




论文级图表已保存至: ../figs\combined_analysis_academic.png


In [ ]:
import optuna
import os

output_dir = '../figs'

fig_imp = optuna.visualization.plot_param_importances(study)

# 3. 映射参数标签 (将代码变量名改为论文显示的名称)
label_map = {
    "lr": "Learning rate",
    "num_models": "Number of models",
    "shrinkage": "Shrinkage"
}

# 获取当前的 y 轴标签（即参数名列表）并进行重命名
# Optuna 的 y 数据通常直接就是 ['shrinkage', 'num_models', 'lr'] 这样的字符串列表
current_labels = fig_imp.data[0].y
new_labels = [label_map.get(label, label) for label in current_labels]

# 直接更新 trace 数据中的 y
fig_imp.update_traces(y=new_labels)

# 4. 视觉与格式美化 (面向论文 4:3)
width = 1000
height = 800

fig_imp.update_layout(
    title=None,
    template='plotly_white',
    xaxis=dict(
        title="Importance Score",
        title_font=dict(size=34, color='black'),
        tickfont=dict(size=32, color='black'),
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside' # 刻度线向外，更学术
    ),
    yaxis=dict(
        title="", 
        tickfont=dict(size=34, color='black'),
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        # 确保 y 轴标签显示完整
        automargin=True 
    ),
    width=width,
    height=height,
    font=dict(family="Arial", color='black'),
)

# 5. 条形图细节优化
fig_imp.update_traces(
    marker_color='#4E79A7',
    marker_line_color='black',
    marker_line_width=1.5,
    opacity=0.85,
    # 在条形图末端显示数值标注（可选，论文常用）
    texttemplate='%{x:.2f}', 
    textposition='inside',
    textfont=dict(size=34)
)

fig_imp.show()

save_path = os.path.join(output_dir, "param_importance.png")
print(f"正在保存高清图片至 {save_path} ...")
fig_imp.write_image(save_path, scale=6)

In [3]:
# 2. Slice Plot (回答“lr 具体设置多少最好？”)
# 既然 lr 最重要，我们需要看它的单变量分布
fig_slice = optuna.visualization.plot_slice(study, params=['lr', 'num_models'])
fig_slice.update_layout(
    title='<b>Hyperparameter Slice Plot</b> (寻找最佳单变量区间)',
    template='plotly_white'
)
fig_slice.update_traces(marker=dict(size=8, opacity=0.7)) # 调整散点大小
print("展示 Slice 分布图...")
fig_slice.show()



展示 Slice 分布图...


In [3]:
import optuna
import os

# 1. 设置路径与标签映射
output_dir = '../figs'
label_map = {
    "lr": "Learning rate",
    "num_models": "Number of models",
    "shrinkage": "Shrinkage"
}

# 2. 生成交互矩阵图 (传入所有三个参数)
# 不指定 params 时默认展示所有，指定则只展示这三个的两两组合
fig_contour = optuna.visualization.plot_contour(study, params=['lr', 'num_models', 'shrinkage'])

# 3. 视觉与格式美化 (面向论文)
width = 1000  # 矩阵图建议宽度大一点
height = 750  # 保持 4:3 左右的比例

fig_contour.update_layout(
    title=None, # 去掉标题
    template='plotly_white',
    width=width,
    height=height,
    font=dict(family="Arial", color='black'),
    # 调整边距以适应多变量标签
    margin=dict(l=80, r=20, t=20, b=80),
)

# 4. 批量修改坐标轴标签
# 因为矩阵图有很多个坐标轴 (xaxis, xaxis2, yaxis, yaxis2...)
# 我们需要遍历所有坐标轴并应用我们的 label_map
for axis_name in fig_contour.layout:
    if axis_name.startswith('xaxis') or axis_name.startswith('yaxis'):
        axis = fig_contour.layout[axis_name]
        # 更新标题字体
        if axis.title.text in label_map:
            axis.title.text = label_map[axis.title.text]
        axis.title.font = dict(size=18, color='black')
        axis.tickfont = dict(size=14, color='black')
        # 增加轴线
        axis.showline = True
        axis.linewidth = 1.5
        axis.linecolor = 'black'
        axis.mirror = True



# 6. 保存高清图片
save_path = os.path.join(output_dir, "param_contour_matrix.png")
try:
    fig_contour.write_image(save_path, scale=3)
    print(f"等高线矩阵图已成功保存至: {save_path}")
except Exception as e:
    print(f"保存失败: {e}")

fig_contour.show()

等高线矩阵图已成功保存至: ../figs\param_contour_matrix.png
